# Batched single-shooting solver comparison

This benchmark compares the experimental custom projected BFGS, Levenberg–Marquardt, and stabilized exact-Newton solvers with SciPy SLSQP. All arms use the same five-day one-room model, parameter bounds, measurements, and float64 objective. Each custom arm uses the canonical model-default start plus seven deterministic samples drawn across the complete normalized lower-to-upper-bound hyperrectangle; the samples are not local perturbations.

## Recorded local CPU screen

A six-hour, five-iteration screen from the same canonical start gave:

- SciPy SLSQP: 3.26 s, objective 20.726
- custom projected BFGS: 7.29 s, objective 35.956
- custom LM: 24.44 s, objective 91.060
- custom exact Newton: 27.08 s, objective 16.344

None converged at this deliberately short cap. The result therefore does **not** select a winner: Newton made the strongest early objective reduction but cost about 8.3x SLSQP wall time; LM was dominated in both time and early quality; BFGS remained behind SLSQP. The A100 multistart run must compare convergence and rollout RMSE, not just five-iteration objective.

Run the short CPU screen first. Only move to the A100 multistart configuration after all derivative and solver checks pass. CUDA setup/capture time is reported inside end-to-end wall time; peak allocated memory is recorded separately.

In [ ]:
# Colab setup. Install the exact branch through pip, then keep a checkout for the runner.
import os, subprocess, sys

REPO = "https://github.com/JBjoernskov/Twin4Build.git"
BRANCH = "feature/issue-128/collocation-initialization"
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
    f"git+{REPO}@{BRANCH}",
], check=True)
if not os.path.exists("/content/Twin4Build"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "/content/Twin4Build"], check=True)
os.chdir("/content/Twin4Build")

import torch
print("torch", torch.__version__)
print("CUDA", torch.cuda.is_available(), torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU")

In [ ]:
# CPU: use HOURS=6, MAXITER=5, N_STARTS=1.
# A100 full run: HOURS=120, MAXITER=300, N_STARTS>=8, BATCH_SIZE chosen after the derivative memory screen.
import json, pathlib, subprocess, sys

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HOURS = 120 if DEVICE == "cuda" else 6
MAXITER = 300 if DEVICE == "cuda" else 5
N_STARTS = 8 if DEVICE == "cuda" else 1
BATCH_SIZE = 4 if DEVICE == "cuda" else 1
ARMS = ["slsqp", "batched-bfgs", "batched-lm", "batched-newton"]

rows = []
for arm in ARMS:
    output = pathlib.Path(f"/content/{arm}_{DEVICE}.json")
    command = [
        sys.executable, "-m", "twin4build.examples.batched_shooting_solver_benchmark",
        "--arm", arm, "--hours", str(HOURS), "--maxiter", str(MAXITER),
        "--device", DEVICE, "--output", str(output),
    ]
    if arm != "slsqp":
        command += ["--n-starts", str(N_STARTS), "--batch-size", str(BATCH_SIZE)]
        if DEVICE == "cuda":
            command += ["--capture"]
    print("Running", arm)
    completed = subprocess.run(command, capture_output=True, text=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        print(f"{arm} FAILED with exit code {completed.returncode}")
        print(completed.stderr[-12000:])
        rows.append({
            "arm": arm, "device": DEVICE, "hardware": torch.cuda.get_device_name() if DEVICE == "cuda" else "CPU",
            "hours": HOURS, "maxiter": MAXITER, "n_starts": 1 if arm == "slsqp" else N_STARTS,
            "seconds": float("nan"), "score_seconds": float("nan"), "success": False,
            "iterations": None, "objective": float("nan"), "rollout_weighted_mse": float("nan"),
            "sensor_rmse": {}, "peak_cuda_memory_gb": float("nan"),
            "error": completed.stderr[-12000:],
        })
        continue
    rows.append(json.loads(output.read_text()))
print("Finished", len(rows), "arms")

In [ ]:
import pandas as pd

summary_columns = [
    "arm", "device", "hardware", "hours", "maxiter", "n_starts",
    "seconds", "score_seconds", "success", "iterations", "objective",
    "rollout_weighted_mse", "sensor_rmse", "peak_cuda_memory_gb",
]
display(pd.DataFrame(rows)[summary_columns])

slsqp = next(row for row in rows if row["arm"] == "slsqp")
for row in rows:
    if row["arm"] == "slsqp":
        continue
    print(
        row["arm"],
        "speedup_vs_slsqp=", slsqp["seconds"] / row["seconds"],
        "objective_delta=", row["objective"] - slsqp["objective"],
    )

print("\nInterpretation rules:")
print("1. Reject an arm that is faster but does not converge or reaches worse rollout quality.")
print("2. Compare cold/end-to-end and warmed derivative costs separately.")
print("3. SLSQP is serial; custom methods report best-of-N and per-start audit data.")
print("4. Repeat at least three start seeds before choosing a production method.")

In [ ]:
derivative_rows = []
for row in rows:
    for bundle, stat in (row.get("derivative_stats") or {}).items():
        derivative_rows.append({
            "arm": row["arm"],
            "bundle": bundle,
            **stat,
            "warmed_mean_seconds": (
                (stat["seconds"] - stat["first_seconds"]) / (stat["calls"] - 1)
                if stat["calls"] > 1 else float("nan")
            ),
        })
if derivative_rows:
    display(pd.DataFrame(derivative_rows))